# Reply Recommendation — LoRA SFT（Colab）

在 **Google Colab** 上微调 **Llama 3.x 3B Instruct**（或同系列），数据为 JSONL（与 `prompts_for_sft` 对齐的字段）。

## 准备
1. **GPU**：运行时 → 更改运行时类型 → **T4 / L4 / A100**。
2. **数据（推荐）**：**训练 / 验证分文件**，例如 `social_2000_merged_train_final.jsonl` + `social_2000_merged_eval_100.jsonl`（`sample_id` 无交集）。若验证路径留空，则从训练文件去重后按比例随机划分验证集。
3. **Hugging Face**：gated 模型需 [Token](https://huggingface.co/settings/tokens) 并在下方登录。

## 本目录
- `train_lora_reply_sft.ipynb`、`requirements.txt`、`prompts_for_sft.py`。Colab 请挂载 Drive 或 `git clone`，必要时填 `SFT_DIR_OVERRIDE`。


In [ ]:
# @title 安装依赖（Colab 每个新运行时执行一次）
# Colab 通常已预装带 CUDA 的 torch，此处不再强制重装 torch，避免环境被破坏。
import sys, subprocess

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q", *packages])

pip_install([
    "transformers>=4.44.0",
    "datasets>=2.19.0",
    "accelerate>=0.33.0",
    "peft>=0.12.0",
    "trl>=0.9.0",
    "sentencepiece",
    "protobuf",
    "bitsandbytes",
    "huggingface_hub",
])

import torch
print("torch", torch.__version__, "cuda?", torch.cuda.is_available())


In [ ]:
# @title 配置路径与 Hugging Face 登录
from pathlib import Path
import os

try:
    from google.colab import drive  # type: ignore
    _IN_COLAB = True
except ImportError:
    drive = None
    _IN_COLAB = False

MOUNT_DRIVE = True  # @param {type:"boolean"}
if _IN_COLAB and MOUNT_DRIVE and drive is not None:
    drive.mount("/content/drive")

from huggingface_hub import login

# 训练集 JSONL（必填）
TRAIN_JSONL = "/content/drive/MyDrive/social_2000_merged_train_final.jsonl"  # @param {type:"string"}
# 验证集 JSONL（推荐填写；留空则从训练集去重后按 VAL_SPLIT_RATIO 随机划分）
EVAL_JSONL = "/content/drive/MyDrive/social_2000_merged_eval_100.jsonl"  # @param {type:"string"}
# 仅当上面 EVAL_JSONL 留空时生效：从单一 TRAIN 去重后随机划出验证集比例与随机种子（已填 EVAL_JSONL 可忽略下面两项）
VAL_SPLIT_RATIO = 0.02  # @param {type:"number"}
VAL_SPLIT_SEED = 42  # @param {type:"integer"}

OUTPUT_DIR = "/content/drive/MyDrive/lora_reply_sft"  # @param {type:"string"}
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"  # @param {type:"string"}

HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)
else:
    login(add_to_git_credential=False)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("TRAIN_JSONL:", TRAIN_JSONL)
print("EVAL_JSONL:", EVAL_JSONL or "(empty → random split from train)")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_ID:", MODEL_ID)


In [ ]:
# @title 构建数据集（train / val）
import importlib
import sys
import json
from pathlib import Path
from datasets import Dataset

CANDIDATE_SFT_DIRS = [
    Path("/content/drive/MyDrive/LA-Hacks/SFT"),
    Path("/content/LA-Hacks/SFT"),
    Path.cwd() / "SFT",
    Path.cwd(),
]
SFT_DIR_OVERRIDE = ""  # @param {type:"string"}
if SFT_DIR_OVERRIDE.strip():
    CANDIDATE_SFT_DIRS.insert(0, Path(SFT_DIR_OVERRIDE.strip()))

sft_dir = next((d for d in CANDIDATE_SFT_DIRS if (d / "prompts_for_sft.py").exists()), None)
if sft_dir is None:
    raise FileNotFoundError(
        "找不到 prompts_for_sft.py。请将仓库中的 SFT 文件夹上传到 Colab（或挂载 Drive），"
        "或在 SFT_DIR_OVERRIDE 填写包含 prompts_for_sft.py 的目录。"
    )
sys.path.insert(0, str(sft_dir))
print("Using SFT dir:", sft_dir)

import prompts_for_sft
importlib.reload(prompts_for_sft)
from prompts_for_sft import record_to_messages


def load_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def dedup_by_sample_id(rows):
    seen = set()
    out = []
    for r in rows:
        sid = r.get("sample_id")
        if sid in seen:
            continue
        seen.add(sid)
        out.append(r)
    return out


def build_example(rec, user_default=None):
    return {
        "messages": record_to_messages(rec, user_default=user_default),
        "sample_id": rec.get("sample_id", ""),
    }


train_rows = dedup_by_sample_id(load_jsonl(TRAIN_JSONL))
print("train rows (after dedup):", len(train_rows))

eval_path = (EVAL_JSONL or "").strip()
if eval_path:
    eval_rows = dedup_by_sample_id(load_jsonl(eval_path))
    print("eval rows (after dedup):", len(eval_rows))
    t_ids = {r.get("sample_id") for r in train_rows}
    e_ids = {r.get("sample_id") for r in eval_rows}
    overlap = t_ids & e_ids
    if overlap:
        raise ValueError(
            f"训练集与验证集存在 {len(overlap)} 个重复 sample_id。示例: {list(overlap)[:5]}"
        )
    train_ds = Dataset.from_list([build_example(r) for r in train_rows])
    eval_ds = Dataset.from_list([build_example(r) for r in eval_rows])
else:
    ds_all = Dataset.from_list([build_example(r) for r in train_rows])
    split = ds_all.train_test_split(test_size=VAL_SPLIT_RATIO, seed=VAL_SPLIT_SEED)
    train_ds, eval_ds = split["train"], split["test"]
    print("val mode: random split from train; eval size:", len(eval_ds))

rows = train_rows
print(train_ds, eval_ds)


In [ ]:
# @title 加载 tokenizer / 模型 + LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

USE_QLORA = False  # @param {type:"boolean"}
if USE_QLORA:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )

model.config.use_cache = False

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# @title 应用 chat template（train / val 各构造 text 列）
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

train_ds_text = train_ds.map(lambda ex: {"text": formatting_func(ex)})
eval_ds_text = eval_ds.map(lambda ex: {"text": formatting_func(ex)})
print("train_ds_text:", len(train_ds_text), "eval_ds_text:", len(eval_ds_text))


In [ ]:
# @title SFTTrainer 训练（train + eval）
from trl import SFTTrainer, SFTConfig

max_seq_length = 1024  # @param {type:"integer"}
# 验证集较小时可适当减小 eval_steps，使验证更频繁
eval_steps = 50  # @param {type:"integer"}

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    packing=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

try:
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_ds_text,
        eval_dataset=eval_ds_text,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_ds_text,
        eval_dataset=eval_ds_text,
        tokenizer=tokenizer,
    )

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter + tokenizer to", OUTPUT_DIR)


## 推理快速检查（可选）

训练结束后可用 `PeftModel.from_pretrained(MODEL_ID, OUTPUT_DIR)` 将 LoRA 挂回基座，对 `record_to_messages` 得到的前两条 message 调用 `apply_chat_template(..., add_generation_prompt=True)`，再 `model.generate`。


In [ ]:
# @title （可选）单条生成烟测
RUN_SMOKE = False  # @param {type:"boolean"}
if RUN_SMOKE and len(rows) > 0:
    from peft import PeftModel
    import torch
    from transformers import AutoModelForCausalLM
    dtype = (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
    dev = next(merged.parameters()).device
    msgs = rows[0]
    chat = record_to_messages(msgs)[:2]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(dev)
    out = merged.generate(**inputs, max_new_tokens=256, do_sample=False)
    print(tokenizer.decode(out[0], skip_special_tokens=False))
